# Perturbation Sensitivity — Do Sinks Limit Information Spread?

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import torch
import numpy as np
import matplotlib.pyplot as plt
import yaml

from src.model import InstrumentedGPS
from src.datasets import get_dataloaders, DATASET_INFO

In [ ]:
def load_experiment(experiment_id, device):
    config_path = f'../outputs/{experiment_id}/config.yaml'
    with open(config_path) as f:
        config = yaml.safe_load(f)
    dataset_info = DATASET_INFO[config['data']['dataset']].copy()
    if config['vnode']['enabled'] and dataset_info.get('num_node_types') is not None:
        dataset_info['num_node_types'] = config['vnode']['num_node_types']
        dataset_info['num_edge_types'] = config['vnode']['num_edge_types']
    model = InstrumentedGPS(config, dataset_info).to(device)
    model.load_state_dict(torch.load(f'../outputs/{experiment_id}/best_model.pt', map_location=device, weights_only=True))
    model.eval()
    return model, config, dataset_info

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Perturbation experiment

In [ ]:
from torch_geometric.utils import degree
from tqdm import tqdm

@torch.no_grad()
def run_perturbation_experiment(model, config, device, max_graphs=100, noise_scale=0.1):
    """Perturb one node per graph, measure how perturbation spreads across layers.
    
    Returns:
        layer_distance_spread: dict[layer] -> dict[distance] -> list of ||delta_h|| values
    """
    _, _, test_loader, _ = get_dataloaders(config)
    model.eval()
    
    num_layers = model.num_layers
    # layer -> distance -> list of perturbation magnitudes
    spread = {l: {} for l in range(num_layers + 1)}
    
    graphs_done = 0
    for batch in tqdm(test_loader, desc='Perturbation'):
        if graphs_done >= max_graphs:
            break
        batch = batch.to(device)
        
        # 1. Clean forward pass
        _ = model(batch, collect_diagnostics=True)
        clean_reps = {l: model.layer_data[l]['h'].clone() for l in range(num_layers + 1)}
        clean_batch_ids = model.layer_data[0]['batch']
        
        unique_graphs = clean_batch_ids.unique()
        
        for g_idx, g_id in enumerate(unique_graphs):
            if graphs_done >= max_graphs:
                break
            
            graph_mask = (clean_batch_ids == g_id)
            node_indices = torch.where(graph_mask)[0]
            num_nodes_g = len(node_indices)
            
            if num_nodes_g < 3:
                continue
            
            # Pick a random node to perturb (not VNode if present)
            perturb_local = torch.randint(0, num_nodes_g, (1,)).item()
            perturb_global = node_indices[perturb_local].item()
            
            # 2. Compute graph distances from perturbed node (BFS)
            # Build adjacency for this graph
            g_edges = batch.edge_index[:, batch.batch[batch.edge_index[0]] == g_id]
            local_map = {int(n): i for i, n in enumerate(node_indices)}
            adj = {i: set() for i in range(num_nodes_g)}
            for s, d in g_edges.t().tolist():
                if s in local_map and d in local_map:
                    adj[local_map[s]].add(local_map[d])
            
            # BFS from perturbed node
            distances = [-1] * num_nodes_g
            distances[perturb_local] = 0
            queue = [perturb_local]
            head = 0
            while head < len(queue):
                curr = queue[head]
                head += 1
                for nbr in adj[curr]:
                    if distances[nbr] == -1:
                        distances[nbr] = distances[curr] + 1
                        queue.append(nbr)
            
            # 3. Perturb and re-run
            batch_perturbed = batch.clone()
            noise = torch.randn_like(batch_perturbed.x[perturb_global:perturb_global+1].float()) * noise_scale
            if batch_perturbed.x.dtype == torch.long:
                # For categorical features, we can't add noise directly
                # Instead, randomly change the node type
                batch_perturbed.x[perturb_global] = torch.randint(0, 28, batch_perturbed.x[perturb_global].shape)
            else:
                batch_perturbed.x[perturb_global] = batch_perturbed.x[perturb_global].float() + noise.squeeze()
            
            _ = model(batch_perturbed, collect_diagnostics=True)
            perturbed_reps = {l: model.layer_data[l]['h'].clone() for l in range(num_layers + 1)}
            
            # 4. Compute perturbation magnitude per node per layer
            for l in range(num_layers + 1):
                clean_h = clean_reps[l][graph_mask]
                perturbed_h = perturbed_reps[l][graph_mask]
                delta = (clean_h - perturbed_h).float().norm(dim=-1)  # (num_nodes,)
                
                for node_i in range(num_nodes_g):
                    dist = distances[node_i]
                    if dist < 0:
                        continue
                    if dist not in spread[l]:
                        spread[l][dist] = []
                    spread[l][dist].append(delta[node_i].item())
            
            graphs_done += 1
    
    return spread

# Run for both models
print("Running perturbation experiment on zinc-novnode-rwse-10L...")
model_novnode, config_novnode, _ = load_experiment('zinc-novnode-rwse-10L', device)
spread_novnode = run_perturbation_experiment(model_novnode, config_novnode, device)

print("\nRunning perturbation experiment on zinc-vnode-rwse-10L...")
model_vnode, config_vnode, _ = load_experiment('zinc-vnode-rwse-10L', device)
spread_vnode = run_perturbation_experiment(model_vnode, config_vnode, device)

## 2. Perturbation heatmaps

In [ ]:
def spread_to_heatmap(spread, max_dist=6):
    """Convert spread dict to a 2D array: (num_layers, max_dist+1)."""
    num_layers = max(spread.keys()) + 1
    heatmap = np.zeros((num_layers, max_dist + 1))
    for l in range(num_layers):
        for d in range(max_dist + 1):
            values = spread[l].get(d, [])
            heatmap[l, d] = np.mean(values) if values else 0.0
    return heatmap

max_dist = 6
hm_novnode = spread_to_heatmap(spread_novnode, max_dist)
hm_vnode = spread_to_heatmap(spread_vnode, max_dist)

# Normalize both to same scale
vmax = max(hm_novnode.max(), hm_vnode.max())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, hm, title in [(axes[0], hm_novnode, 'Without VNode'), (axes[1], hm_vnode, 'With VNode')]:
    im = ax.imshow(hm.T, aspect='auto', origin='lower', cmap='YlOrRd', vmin=0, vmax=vmax)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Graph distance from perturbed node')
    ax.set_title(title)
    ax.set_yticks(range(max_dist + 1))

plt.colorbar(im, ax=axes, label='Mean perturbation magnitude ||delta h||')
plt.suptitle('Figure 4: Perturbation Sensitivity — How Far Does a Perturbation Spread?', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figure4_perturbation_heatmap.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 3. Perturbation spread at distance 0 vs distance 3+ across layers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, spread, title in [(axes[0], spread_novnode, 'Without VNode'), (axes[1], spread_vnode, 'With VNode')]:
    num_layers = max(spread.keys()) + 1
    for dist, color, label in [(0, 'tab:red', 'Distance 0 (perturbed)'),
                                (1, 'tab:orange', 'Distance 1'),
                                (2, 'tab:blue', 'Distance 2'),
                                (3, 'tab:green', 'Distance 3+')]:
        values = []
        for l in range(num_layers):
            if dist < 3:
                vals = spread[l].get(dist, [])
            else:
                vals = []
                for d in range(3, max_dist + 1):
                    vals.extend(spread[l].get(d, []))
            values.append(np.mean(vals) if vals else 0.0)
        ax.plot(range(num_layers), values, color=color, linewidth=2, label=label, marker='o', markersize=3)
    
    ax.set_xlabel('Layer')
    ax.set_ylabel('Mean perturbation magnitude')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figure4b_perturbation_by_distance.pdf', bbox_inches='tight', dpi=150)
plt.show()